# Importar codigos

Se obtienen las funciones del codigo

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import subprocess
import math
import random



In [2]:
import import_ipynb
from Recodificacion import *
from Codigos import *

In [3]:
METODOS      = ["pillow", "ffmpeg", "imagemagick"]
FORMATOS     = ["jpg", "webp", "png"]
CARPETA_RAIZ = "imagenes"
CALIDADES_POOL = [10, 30, 50, 70, 90]
CALIDADES_POR_IMAGEN = 3
SEED = 42  # reproducible — mismas calidades siempre para la misma imagen


# Funcion para obtener todos los valores cuantitativos de la imagen

Obtiene los valores seleccionados de la parte teorica

In [4]:
import os

def recodificar_imagen(
    img_path,
    metodo,
    formato,
    calidad
):

    # -----------------------------------------
    # VALIDACIONES
    # -----------------------------------------

    metodo = metodo.lower()
    formato = formato.lower()

    metodos_validos = [
        "pillow",
        "ffmpeg",
        "imagemagick"
    ]

    formatos_validos = [
        "jpg",
        "png",
        "webp"
    ]

    if metodo not in metodos_validos:
        raise ValueError(
            f"Método no soportado: {metodo}"
        )

    if formato not in formatos_validos:
        raise ValueError(
            f"Formato no soportado: {formato}"
        )

    # -----------------------------------------
    # RUTA SALIDA
    # -----------------------------------------

    nombre = os.path.splitext(
        os.path.basename(img_path)
    )[0]

    out_dir = os.path.join(
        "imagenes",
        metodo,
        formato
    )

    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(
        out_dir,
        f"{nombre}_{metodo}_q{calidad}.{formato}"
    )

    # -----------------------------------------
    # LLAMAR FUNCIÓN CORRESPONDIENTE
    # -----------------------------------------

    if metodo == "pillow":

        convert_image_pillow(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    elif metodo == "ffmpeg":

        convert_with_ffmpeg(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    elif metodo == "imagemagick":

        convert_with_imagemagick(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    # -----------------------------------------
    # VALIDAR GENERACIÓN
    # -----------------------------------------

    if not os.path.exists(out_path):

        raise RuntimeError(
            f"No se generó el archivo:\n{out_path}"
        )

    # -----------------------------------------
    # RETORNO
    # -----------------------------------------

    return out_path

In [5]:
def extraer_features(path, prefix):
    f = {}

    # básicas
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    # contraste
    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)

    # frecuencia
    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    # bordes multi escala
    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    # gradiente
    f[f"{prefix}_grad_mean"] = obtener_contraste_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)

    # textura (GLCM)
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_glcm(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_glcm(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    # entropía
    f[f"{prefix}_entropia_shannon"] = obtener_entropia_shannon(path)
    f[f"{prefix}_entropia_espectral"] = obtener_entropia_espectral(path)
    f[f"{prefix}_entropia_gradiente"] = obtener_entropia_gradiente(path)
    f[f"{prefix}_entropia_glcm"] = obtener_entropia_glcm(path)

    # color
    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)

    return f

In [11]:
def procesar_imagen(
    img_path,
    df,
    metodo="pillow",
    formato="jpg",
    calidad=50
):
    import pandas as pd

    if not isinstance(df, pd.DataFrame):
        raise TypeError("df debe ser un pandas.DataFrame")

    # 1. recodificar
    out_path = recodificar_imagen(img_path, metodo, formato, calidad)

    # 2. features
    feats_orig = extraer_features(img_path, "orig")
    feats_comp = extraer_features(out_path, "comp")

    # 3. métrica objetivo
    metricas = {"lpips": calcular_lpips(img_path, out_path)}

    # 4. fila
    fila = {
        "img_original":     img_path,
        "img_recodificada": out_path,
        "metodo":           metodo,
        "formato":          formato,
        "calidad":          calidad,
        **feats_orig,
        **feats_comp,
        **metricas
    }

    return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)

In [8]:
def procesar_dataset_completo(
    df: pd.DataFrame,
    formato_origen: str = "jpg",
    inicio: int = 0,
    fin: int = None
) -> pd.DataFrame:

    random.seed(SEED)

    formato_orig = formato_origen.lower().strip()

    if formato_orig not in FORMATOS:
        raise ValueError(
            f"formato_origen debe ser uno de {FORMATOS}, recibido: '{formato_orig}'"
        )

    carpeta = os.path.join(CARPETA_RAIZ, formato_orig)

    if not os.path.isdir(carpeta):
        raise FileNotFoundError(f"Carpeta no encontrada: {carpeta}")

    imagenes = sorted([
        f for f in os.listdir(carpeta)
        if f.lower().endswith(f".{formato_orig}")
    ])

    if not imagenes:
        print(f"⚠ Sin imágenes en: {carpeta}")
        return df

    formatos_salida = [f for f in FORMATOS if f != formato_orig]
    fmt_A = formatos_salida[0]
    fmt_B = formatos_salida[1]

    mitad = math.ceil(len(imagenes) / 2)
    mitad_A = imagenes[:mitad]
    mitad_B = imagenes[mitad:]

    todas = list(zip(
        mitad_A + mitad_B,
        [fmt_A] * len(mitad_A) + [fmt_B] * len(mitad_B)
    ))

    fin_real = fin if fin is not None else len(todas)
    subset = todas[inicio:fin_real]

    print(f"\n{'='*60}")
    print(f"📁 {carpeta}  ({len(imagenes)} imágenes total)")
    print(f"   Procesando [{inicio}:{fin_real}] → {len(subset)} imágenes")
    print(f"{'='*60}")

    for idx_global, (nombre_archivo, formato_sal) in enumerate(
        subset,
        start=inicio
    ):

        img_path = os.path.join(carpeta, nombre_archivo)

        metodo = METODOS[idx_global % len(METODOS)]

        # Una sola calidad aleatoria por imagen
        calidad = random.choice(CALIDADES_POOL)

        print(
            f"\n🖼 [{idx_global}] {nombre_archivo}"
            f" → {formato_sal}"
            f" [{metodo}]"
            f" calidad={calidad}"
        )

        try:
            df = procesar_imagen(
                img_path=img_path,
                df=df,
                metodo=metodo,
                formato=formato_sal,
                calidad=calidad
            )

        except Exception as e:
            print(f"❌ q={calidad} {e}")
            continue

    print(
        f"\n✅ [{inicio}:{fin_real}] completado "
        f"— filas acumuladas: {len(df)}"
    )

    return df

# Metodo objetivo

In [9]:
import torch
import torch.nn.functional as F
import numpy as np
from skimage import io
from skimage.util import img_as_ubyte
import lpips

# ---------------------------
# CONFIGURACIÓN
# ---------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn_lpips = lpips.LPIPS(net='alex').to(device)  # red elegida: AlexNet (mejor correlación en BAPPS, Zhang et al., 2018)

# ---------------------------
# VALIDACIÓN DE FORMATO (sin conversión)
# ---------------------------

def es_rgb_valida(img):
    """
    Verifica si una imagen ya está en formato RGB de 3 canales,
    sin realizar ninguna conversión.

    Retorna
    -------
    bool
        True si la imagen es RGB (3 canales), False en cualquier
        otro caso (escala de grises, RGBA, u otro formato).
    """
    return img.ndim == 3 and img.shape[-1] == 3

def load_tensor(path, device):
    """
    Carga una imagen y la convierte a tensor solo si ya está en
    formato RGB. Si no lo está (escala de grises, RGBA, etc.),
    retorna None para que sea omitida.

    Retorna
    -------
    torch.Tensor o None
        Tensor listo para LPIPS, o None si la imagen debe omitirse.
    """
    img = io.imread(path)

    if not es_rgb_valida(img):
        return None

    img = img_as_ubyte(img)
    img = img.astype(np.float32) / 127.5 - 1.0
    return torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)

def match_size(img1, img2):
    if img1.shape != img2.shape:
        img2 = F.interpolate(
            img2,
            size=img1.shape[2:],
            mode='bilinear',
            align_corners=False
        )
    return img1, img2

# ---------------------------
# LPIPS
# ---------------------------

def calcular_lpips(img_orig_path, img_comp_path):
    """
    Calcula la distancia perceptual LPIPS entre dos imágenes.

    Si alguna de las dos imágenes no está en formato RGB (por
    ejemplo, escala de grises o RGBA), la comparación se omite
    y se retorna None en lugar de forzar una conversión.

    Parámetros
    ----------
    img_orig_path : str
        Ruta de la imagen original.
    img_comp_path : str
        Ruta de la imagen comprimida o transformada.

    Retorna
    -------
    float o None
        Distancia perceptual LPIPS, o None si alguna imagen
        fue omitida por no ser RGB.
    """
    orig = load_tensor(img_orig_path, device)
    comp = load_tensor(img_comp_path, device)

    if orig is None or comp is None:
        return None

    orig, comp = match_size(orig, comp)

    with torch.no_grad():
        dist = loss_fn_lpips(orig, comp)

    return dist.item()

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: d:\TrabajoGrado\Codigos\Funciones_Matematicas\Generar_Dataset\lib\site-packages\lpips\weights\v0.1\alex.pth


In [12]:
# Celda 1 — se ejecuta, usa RAM, termina y libera
df = pd.DataFrame()
df = procesar_dataset_completo(df, formato_origen="jpg")
df.to_csv("csv/dataset_jpg.csv", index=False)
del df  # fuerza la liberación inmediata
print("✅ JPG listo")


📁 imagenes\jpg  (10000 imágenes total)
   Procesando [0:10000] → 10000 imágenes

🖼 [0] COCO_test2014_000000000057.jpg → webp [pillow] calidad=10
Convertido: imagenes\jpg\COCO_test2014_000000000057.jpg → imagenes\pillow\webp\COCO_test2014_000000000057_pillow_q10.webp

🖼 [1] COCO_test2014_000000000080.jpg → webp [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\webp\COCO_test2014_000000000080_ffmpeg_q10.webp

🖼 [2] COCO_test2014_000000000090.jpg → webp [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\webp\COCO_test2014_000000000090_imagemagick_q50.webp

🖼 [3] COCO_test2014_000000000155.jpg → webp [pillow] calidad=30
Convertido: imagenes\jpg\COCO_test2014_000000000155.jpg → imagenes\pillow\webp\COCO_test2014_000000000155_pillow_q30.webp

🖼 [4] COCO_test2014_000000000171.jpg → webp [ffmpeg] calidad=30
[FFmpeg] Convertido → imagenes\ffmpeg\webp\COCO_test2014_000000000171_ffmpeg_q30.webp

🖼 [5] COCO_test2014_000000000173.jpg → webp [imagemagick] calidad=30
[I

In [13]:
# Celda 2 — arranca con RAM limpia
df = pd.read_csv("csv/dataset_jpg.csv")
df = procesar_dataset_completo(df, formato_origen="webp")
df.to_csv("csv/dataset_jpg_webp.csv", index=False)
del df
print("✅ WEBP listo")


📁 imagenes\webp  (10000 imágenes total)
   Procesando [0:10000] → 10000 imágenes

🖼 [0] 0 (2).webp → jpg [pillow] calidad=10
Convertido: imagenes\webp\0 (2).webp → imagenes\pillow\jpg\0 (2)_pillow_q10.jpg

🖼 [1] 0.webp → jpg [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\0_ffmpeg_q10.jpg

🖼 [2] 0044e5aa-e000-4090-8030-00ff5135e73c.webp → jpg [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\jpg\0044e5aa-e000-4090-8030-00ff5135e73c_imagemagick_q50.jpg

🖼 [3] 006c6de7-a9d9-402e-a2ef-094f19f299ba.webp → jpg [pillow] calidad=30
Convertido: imagenes\webp\006c6de7-a9d9-402e-a2ef-094f19f299ba.webp → imagenes\pillow\jpg\006c6de7-a9d9-402e-a2ef-094f19f299ba_pillow_q30.jpg

🖼 [4] 008c56e6-c177-4ab0-a58c-6fd1e0b106dc.webp → jpg [ffmpeg] calidad=30
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\008c56e6-c177-4ab0-a58c-6fd1e0b106dc_ffmpeg_q30.jpg

🖼 [5] 00a99d2c-01d9-454e-9f7d-26bc066282fc.webp → jpg [imagemagick] calidad=30
[ImageMagick] Convertido → imagenes\imag

In [14]:
# Bloque 1 — primeras 500
df = pd.read_csv("csv/dataset_jpg_webp.csv")
df = procesar_dataset_completo(df, formato_origen="png")
df.to_csv("csv/dataset_completo.csv", index=False)
del df


📁 imagenes\png  (10000 imágenes total)
   Procesando [0:10000] → 10000 imágenes

🖼 [0] 00000000_224resized.png → jpg [pillow] calidad=10
Convertido: imagenes\png\00000000_224resized.png → imagenes\pillow\jpg\00000000_224resized_pillow_q10.jpg

🖼 [1] 00000000_300resized.png → jpg [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000000_300resized_ffmpeg_q10.jpg

🖼 [2] 00000000_512resized (2).png → jpg [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000000_512resized (2)_imagemagick_q50.jpg

🖼 [3] 00000001_224resized.png → jpg [pillow] calidad=30
Convertido: imagenes\png\00000001_224resized.png → imagenes\pillow\jpg\00000001_224resized_pillow_q30.jpg

🖼 [4] 00000001_300resized (2).png → jpg [ffmpeg] calidad=30
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000001_300resized (2)_ffmpeg_q30.jpg

🖼 [5] 00000001_512resized.png → jpg [imagemagick] calidad=30
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000001_512resized_imagemagick_q30.jpg



C:\Users\santi\AppData\Local\Temp\ipykernel_8132\1980866941.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)



🖼 [315] 00000099_512resized.png → jpg [pillow] calidad=30
Convertido: imagenes\png\00000099_512resized.png → imagenes\pillow\jpg\00000099_512resized_pillow_q30.jpg

🖼 [316] 00000100_224resized.png → jpg [ffmpeg] calidad=50
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000100_224resized_ffmpeg_q50.jpg

🖼 [317] 00000100_512resized (2).png → jpg [imagemagick] calidad=70
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000100_512resized (2)_imagemagick_q70.jpg

🖼 [318] 00000100_512resized.png → jpg [pillow] calidad=50
Convertido: imagenes\png\00000100_512resized.png → imagenes\pillow\jpg\00000100_512resized_pillow_q50.jpg

🖼 [319] 00000101_224resized.png → jpg [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000101_224resized_ffmpeg_q10.jpg

🖼 [320] 00000101_300resized.png → jpg [imagemagick] calidad=10
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000101_300resized_imagemagick_q10.jpg

🖼 [321] 00000101_512resized (2).png → jpg [pillow] calidad=70
Convertido: ima

C:\Users\santi\AppData\Local\Temp\ipykernel_8132\1980866941.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)



🖼 [1056] 00000404_512resized.png → jpg [pillow] calidad=10
Convertido: imagenes\png\00000404_512resized.png → imagenes\pillow\jpg\00000404_512resized_pillow_q10.jpg

🖼 [1057] 00000405_300resized.png → jpg [ffmpeg] calidad=90
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000405_300resized_ffmpeg_q90.jpg

🖼 [1058] 00000405_512resized (2).png → jpg [imagemagick] calidad=10
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000405_512resized (2)_imagemagick_q10.jpg

🖼 [1059] 00000405_512resized.png → jpg [pillow] calidad=30
Convertido: imagenes\png\00000405_512resized.png → imagenes\pillow\jpg\00000405_512resized_pillow_q30.jpg

🖼 [1060] 00000406_300resized (2).png → jpg [ffmpeg] calidad=50
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\00000406_300resized (2)_ffmpeg_q50.jpg

🖼 [1061] 00000406_300resized.png → jpg [imagemagick] calidad=30
[ImageMagick] Convertido → imagenes\imagemagick\jpg\00000406_300resized_imagemagick_q30.jpg

🖼 [1062] 00000406_512resized (2).png → jpg [pillow] calidad=10


...


🖼 [4602] Cats_Test1836.png → jpg [pillow] calidad=70
Convertido: imagenes\png\Cats_Test1836.png → imagenes\pillow\jpg\Cats_Test1836_pillow_q70.jpg

🖼 [4603] Cats_Test1837.png → jpg [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\Cats_Test1837_ffmpeg_q10.jpg

🖼 [4604] Cats_Test1838.png → jpg [imagemagick] calidad=70
[ImageMagick] Convertido → imagenes\imagemagick\jpg\Cats_Test1838_imagemagick_q70.jpg

🖼 [4605] Cats_Test1839.png → jpg [pillow] calidad=50
Convertido: imagenes\png\Cats_Test1839.png → imagenes\pillow\jpg\Cats_Test1839_pillow_q50.jpg

🖼 [4606] Cats_Test184.png → jpg [ffmpeg] calidad=90
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\Cats_Test184_ffmpeg_q90.jpg

🖼 [4607] Cats_Test1840.png → jpg [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\jpg\Cats_Test1840_imagemagick_q50.jpg

🖼 [4608] Cats_Test1841.png → jpg [pillow] calidad=10
Convertido: imagenes\png\Cats_Test1841.png → imagenes\pillow\jpg\Cats_Test1841_pillow_q10.jpg

🖼 [4609] Cats_Tes

C:\Users\santi\AppData\Local\Temp\ipykernel_8132\1980866941.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)



🖼 [4611] Cats_Test1846.png → jpg [pillow] calidad=90
Convertido: imagenes\png\Cats_Test1846.png → imagenes\pillow\jpg\Cats_Test1846_pillow_q90.jpg

🖼 [4612] Cats_Test1848.png → jpg [ffmpeg] calidad=90
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\Cats_Test1848_ffmpeg_q90.jpg

🖼 [4613] Cats_Test1849.png → jpg [imagemagick] calidad=70
[ImageMagick] Convertido → imagenes\imagemagick\jpg\Cats_Test1849_imagemagick_q70.jpg

🖼 [4614] Cats_Test185.png → jpg [pillow] calidad=70
Convertido: imagenes\png\Cats_Test185.png → imagenes\pillow\jpg\Cats_Test185_pillow_q70.jpg

🖼 [4615] Cats_Test1850.png → jpg [ffmpeg] calidad=10
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\Cats_Test1850_ffmpeg_q10.jpg

🖼 [4616] Cats_Test1851.png → jpg [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\jpg\Cats_Test1851_imagemagick_q50.jpg

🖼 [4617] Cats_Test1854.png → jpg [pillow] calidad=30
Convertido: imagenes\png\Cats_Test1854.png → imagenes\pillow\jpg\Cats_Test1854_pillow_q30.jpg

🖼 [4618] Cats_Test

C:\Users\santi\AppData\Local\Temp\ipykernel_8132\1980866941.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)



🖼 [4995] Cats_Test2295.png → jpg [pillow] calidad=70
Convertido: imagenes\png\Cats_Test2295.png → imagenes\pillow\jpg\Cats_Test2295_pillow_q70.jpg

🖼 [4996] Cats_Test2296.png → jpg [ffmpeg] calidad=30
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\Cats_Test2296_ffmpeg_q30.jpg

🖼 [4997] Cats_Test2297.png → jpg [imagemagick] calidad=50
[ImageMagick] Convertido → imagenes\imagemagick\jpg\Cats_Test2297_imagemagick_q50.jpg

🖼 [4998] Cats_Test2299.png → jpg [pillow] calidad=70
Convertido: imagenes\png\Cats_Test2299.png → imagenes\pillow\jpg\Cats_Test2299_pillow_q70.jpg

🖼 [4999] Cats_Test230.png → jpg [ffmpeg] calidad=50
[FFmpeg] Convertido → imagenes\ffmpeg\jpg\Cats_Test230_ffmpeg_q50.jpg

🖼 [5000] Cats_Test2300.png → webp [imagemagick] calidad=10
[ImageMagick] Convertido → imagenes\imagemagick\webp\Cats_Test2300_imagemagick_q10.webp

🖼 [5001] Cats_Test2301.png → webp [pillow] calidad=30
Convertido: imagenes\png\Cats_Test2301.png → imagenes\pillow\webp\Cats_Test2301_pillow_q30.webp

🖼 [5002] Ca

In [2]:
import os
import pandas as pd

df = pd.read_csv("csv/dataset_completo.csv")

# 1. Renombrar 'formato' -> 'formato_salida'
dataset_final = df.rename(columns={"formato": "formato_salida"})

# 2. Extraer el formato de entrada desde 'img_original' (extensión después del último '.')
dataset_final["formato_entrada"] = dataset_final["img_original"].apply(
    lambda ruta: os.path.splitext(ruta)[1].lstrip(".").lower()
)

dataset_final

,img_original,img_recodificada,metodo,formato_salida,calidad,orig_rango,orig_std,orig_michelson,orig_hf,orig_lf,...,comp_glcm_energy,comp_glcm_corr,comp_entropia_shannon,comp_entropia_espectral,comp_entropia_gradiente,comp_entropia_glcm,comp_sat_mean,comp_sat_var,lpips,formato_entrada
0,imagenes\jpg\COCO_test2014_000000000057.jpg,imagenes\pillow\webp\COCO_test2014_00000000005...,pillow,webp,10,1000000.000000,0.099194,0.543779,0.005543,0.994457,...,0.106159,0.935549,5.873461,0.363928,3.881871,8.356802,0.767468,0.013908,0.401984,jpg
1,imagenes\jpg\COCO_test2014_000000000080.jpg,imagenes\ffmpeg\webp\COCO_test2014_00000000008...,ffmpeg,webp,10,997194.509804,0.219470,0.880256,0.009199,0.990801,...,0.021662,0.972981,7.474851,1.668115,5.720261,12.158096,0.165906,0.026545,0.190529,jpg
2,imagenes\jpg\COCO_test2014_000000000090.jpg,imagenes\imagemagick\webp\COCO_test2014_000000...,imagemagick,webp,50,508.824615,0.181978,0.686133,0.003292,0.996708,...,0.053102,0.963162,7.413015,0.634819,4.397896,10.489679,0.252161,0.083100,0.129367,jpg
3,imagenes\jpg\COCO_test2014_000000000155.jpg,imagenes\pillow\webp\COCO_test2014_00000000015...,pillow,webp,30,1000000.000000,0.289459,0.828212,0.002071,0.997929,...,0.046482,0.995003,7.663089,1.439062,4.604019,10.949440,0.380688,0.083645,0.142226,jpg
4,imagenes\jpg\COCO_test2014_000000000171.jpg,imagenes\ffmpeg\webp\COCO_test2014_00000000017...,ffmpeg,webp,30,1000000.000000,0.134413,1.000000,0.037115,0.962885,...,0.360830,0.938405,4.151036,6.663704,2.111251,5.396500,0.271691,0.107001,0.072366,jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29994,imagenes\png\Image_99 (4).png,imagenes\imagemagick\webp\Image_99 (4)_imagema...,imagemagick,webp,10,5.036773,0.207477,0.317264,0.000239,0.999761,...,0.562050,0.994439,3.381717,0.396933,2.089863,4.242423,0.174508,0.052665,0.095534,png
29995,imagenes\png\Image_99 (5).png,imagenes\pillow\webp\Image_99 (5)_pillow_q50.webp,pillow,webp,50,22.631151,0.214202,0.547744,0.001426,0.998574,...,0.340908,0.981175,4.849703,0.622106,2.729117,6.215894,0.201606,0.048831,0.055525,png
29996,imagenes\png\Image_99 (6).png,imagenes\ffmpeg\webp\Image_99 (6)_ffmpeg_q10.webp,ffmpeg,webp,10,1000000.000000,0.349980,0.893117,0.004545,0.995455,...,0.201707,0.968752,6.516667,1.326323,4.768874,9.755698,0.164777,0.032411,0.110600,png
29997,imagenes\png\Image_99 (7).png,imagenes\imagemagick\webp\Image_99 (7)_imagema...,imagemagick,webp,50,1000000.000000,0.399209,0.949520,0.003846,0.996154,...,0.497092,0.983582,4.545579,1.626167,4.244737,7.446053,0.292490,0.118659,0.040584,png


In [3]:
print(dataset_final["formato_entrada"].value_counts())

formato_entrada
webp    10000
png     10000
jpg      9999
Name: count, dtype: int64


In [5]:
dataset_final.to_csv("csv/dataset_final.csv", index=False)